# MNIST Inference with OpenEye and Tensorflow/Tensorflow Lite
This notebook demonstrates how to simulate the OpenEye hardware accelerator and
the accompanying development tools to simulate the inference on the MNIST
dataset using a Tensorflow model.

It covers training a simple neural network, quantizing and exporting it using
Tensorflow Lite. Subsequently, we use OpenEye to simulate the inference on an image
of the test dataset.

First, we do the imports:

In [ ]:
# set the notebook so that it alwys re-imports modules wenn they are imported
%load_ext autoreload
%autoreload 2

import os
import numpy as np
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow logging
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist

from tensorflow.keras.utils import to_categorical

# Load and preprocess the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.reshape((x_train.shape[0], 28, 28, 1)).astype('float32') / 255
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1)).astype('float32') / 255
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

Now, we define the DNN:

In [ ]:

# train the model only if it does not exist
if os.path.exists('mnist_cnn_model.h5'):
    model = tf.keras.models.load_model('mnist_cnn_model.h5')
    print("Model loaded from mnist_cnn_model.h5")                                                           
else:

    # Build a Convolutional Neural Network (CNN) architecture
    model = models.Sequential([
        # First convolutional layer: 32 filters, 3x3 kernel, ReLU activation
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        # First pooling layer: reduces spatial dimensions by half
        layers.MaxPooling2D((2, 2)),
        
        # Second convolutional layer: 64 filters, 3x3 kernel
        layers.Conv2D(64, (3, 3), activation='relu'),
        # Second pooling layer
        layers.MaxPooling2D((2, 2)),
        
        # Third convolutional layer: 64 filters, 3x3 kernel
        layers.Conv2D(64, (3, 3), activation='relu'),
        
        # Flatten the 2D feature maps to 1D for the dense layers
        layers.Flatten(),
        # Fully connected layer with 64 neurons
        layers.Dense(64, activation='relu'),
        # Output layer with 10 neurons (one per digit class), softmax for probabilities
        layers.Dense(10, activation='softmax')
    ])

    # Compile the model with optimizer, loss function, and metrics
    model.compile(optimizer='adam',
                loss='categorical_crossentropy',
                metrics=['accuracy'])

    # Train the model on the training data
    # - 5 epochs: number of times to iterate over the entire dataset
    # - batch_size=64: number of samples processed before updating weights
    # - validation_split=0.1: use 10% of training data for validation
    model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.1)

    # Evaluate the trained model on the test set
    test_loss, test_acc = model.evaluate(x_test, y_test)
    print(f"Test accuracy: {test_acc:.4f}")

    # Save the trained model to a file for later use
    model.save('mnist_cnn_model.h5')

    print("Model trained and saved to mnist_cnn_model.h5")

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")

## Step 3: Convert to TensorFlow Lite with INT8 Quantization

To run the model on hardware accelerators like OpenEye, we need to quantize it. This step:
- Converts the trained model to TensorFlow Lite format
- Applies INT8 quantization to reduce model size and improve inference speed
- Uses a representative dataset to calibrate the quantization process
- Saves the quantized model to a .tflite file

Quantization converts 32-bit floating point weights and activations to 8-bit integers, making the model more suitable for hardware deployment.

In [ ]:
# Number of samples to use for representative dataset during quantization
rep_ds_size = 100

# Initialize TensorFlow Lite converter with the trained Keras model
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable default optimizations (quantization)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Specify that we want INT8 operations in the quantized model
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Set the data type for weights and activations to INT8
converter.target_spec.supported_types = [tf.int8]

def representative_dataset_gen():
    """
    Generator function for post-training quantization calibration.
    Provides sample inputs to determine the range of activations for quantization.
    """
    # Use the first rep_ds_size samples from training data
    for i in range(rep_ds_size):
        # Add batch dimension: (28, 28, 1) -> (1, 28, 28, 1)
        sample = np.expand_dims(x_train[i], axis=0).astype(np.float32)
        yield [sample]

# Assign the representative dataset for quantization calibration
converter.representative_dataset = representative_dataset_gen

# Convert the model to TensorFlow Lite format with INT8 quantization
tflite_model = converter.convert()

# Save the quantized model to disk
model_name = "mnist_quantized_model"

# set the model path as an absolute path 
model_path = os.path.abspath(model_name + '.tflite')
with open(model_path, 'wb') as f:
    f.write(tflite_model)

print(f"Quantisiertes Modell gespeichert als: {model_path}")


Now we do a quick check to compare the outputs of the original and the quantized DNN:

In [ ]:
# Load the quantized TFLite model into an interpreter
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

# Get information about input and output tensors
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Display tensor information to verify quantization
print(f"Input shape: {input_details[0]['shape']}")
print(f"Input type: {input_details[0]['dtype']}")
print(f"Output shape: {output_details[0]['shape']}")
print(f"Output type: {output_details[0]['dtype']}")

# Test inference with a single sample from the test set
test_sample = x_test[0:1].astype(np.float32)

# Set the input tensor and run inference
interpreter.set_tensor(input_details[0]['index'], test_sample)
interpreter.invoke()

# Get the output tensor (predicted probabilities)
tflite_result = interpreter.get_tensor(output_details[0]['index'])

# Compare predictions between original and quantized models
print(f"Original prediction: {np.argmax(model.predict(test_sample))}")
print(f"TFLite prediction: {np.argmax(tflite_result)}")

## Step 5: Set Up OpenEye Hardware Simulation

Now we'll simulate the OpenEye FPGA hardware accelerator using Cocotb (a coroutine-based co-simulation testbench environment).

This section:
- Imports the necessary testing frameworks (pytest and cocotb)
- Sets up paths to the OpenEye HDL (Hardware Description Language) files
- Configures the test environment with access to the testbench and utilities

Cocotb allows us to verify that our quantized model will run correctly on the OpenEye hardware before actual deployment.

In [ ]:
# Import testing frameworks for hardware simulation
import pytest
import cocotb_test
import cocotb_test.simulator
import os, sys

# Get the base directory of the OpenEye project
openeye_base = (os.path.abspath(os.path.join(os.pardir, os.pardir)))

# Set up paths to test directories
tests_dir = os.path.abspath(os.path.join(openeye_base, "test"))
tb_dir = os.path.join(tests_dir, "cocotb_fpga")  # Testbench directory

# Set up paths to HDL (Hardware Description Language) files
hdl_dir = os.path.join(os.path.join(openeye_base, "hdl"))
default_headers_dir = os.path.join(hdl_dir, "include")

# Add test directory to Python path for importing test utilities
sys.path.append(tests_dir)

print(tests_dir)

# Import test utilities for OpenEye simulation
import test_utils as tu


## Step 6: Configure Simulation Clock Parameters

Define the timing parameters for the FPGA simulation. These parameters control:
- **Clock cycle**: The period of the main clock signal
- **Clock delays**: Input and output timing delays to simulate real hardware behavior

These timing parameters ensure accurate simulation of the OpenEye hardware accelerator.

In [ ]:
# Main clock cycle period and unit
clk_cycle = 20          # Clock period value
clk_cycle_unit = "ns"   # Nanoseconds

# Input clock delay (setup time)
clk_delay_in = 100       # Delay value
clk_delay_unit_in = "ps" # Picoseconds

# Output clock delay (hold time)
clk_delay_out = 100        # Delay value
clk_delay_unit_out = "ps"  # Picoseconds

Now we define the DUT and the module that we want to simulate:

In [ ]:
# Define the Device Under Test (DUT) - the OpenEye FPGA module
dut = 'OpenEye_FPGA'

# Define the testbench module that will test the DUT
module = 'OpenEye_FPGA_tb'

# Set the top-level module for simulation
toplevel = dut

In [ ]:
# Import the main test utilities module
import test_utils.test_utils_main

# Automatically gather all Verilog source files from the HDL directory
verilog_sources = test_utils.test_utils_main.get_verilog_sources(hdl_dir)

# Create a target directory for this specific model's simulation results
target_dir = os.path.join(tests_dir, 'simulation/' + model_name)

In [ ]:
# Import nest_asyncio to enable nested event loops
import nest_asyncio

# Apply the patch to allow Cocotb's async operations in Jupyter
nest_asyncio.apply()

In [ ]:
# Import OpenEye parameter management utilities
import test_utils.open_eye_parameters as oe_params

# Import Verilog header file creator for hardware configuration
import test_utils.vh_file_creator as vh_file_creator
# Import register map generator utilities
import test_utils.generator as generator

We need to generate some files based on the configuration of the OpenEye:

In [ ]:
# Create an OpenEye parameters object with default configuration
oep = oe_params.OpenEyeParameters()

# Generate a Verilog header file with the hardware parameters
# This file will be included in the HDL during simulation
vh_file_creator.create_vh_file(oep, os.path.join(os.getcwd(), "parameters.vh"))

# We also need to create a regmap_params.vh file for the register map
generator.create_regmap_params_vh_file(os.path.join(hdl_dir, "config"))


In [ ]:
print(model_path)

## Step 14: Run the Hardware Simulation

Execute the Cocotb simulation to test the quantized MNIST model on the OpenEye FPGA simulator.

This step:
- Configures the Icarus Verilog simulator with all necessary files and parameters
- Runs the `model_test` testcase from the testbench
- Passes the quantized model path and timing parameters to the simulation
- Generates waveform files for debugging (waves=True)
- Stores simulation results in the target directory

The simulation verifies that the quantized model runs correctly on the OpenEye hardware accelerator.

In [ ]:
# Run the Cocotb hardware simulation
results = cocotb_test.simulator.run(
    # Python module search paths for testbench code
    python_search=[tb_dir],
    
    # List of all Verilog source files to compile
    verilog_sources=verilog_sources,
    
    # Top-level module to simulate
    toplevel=toplevel,
    
    # Testbench module containing the test
    module=module,
    
    # Directory where simulation build artifacts are stored
    sim_build=target_dir,
    
    # Specific test case to run
    testcase='model_test',
    
    # Set to True to force recompilation of Verilog sources
    force_compile=False,
    
    # Enable waveform generation for debugging
    waves=True,
    
    # Include directories for Verilog header files
    includes=[hdl_dir, default_headers_dir, os.getcwd()],
    
    # Simulator to use (Icarus Verilog)
    simulator="icarus",
    
    # Environment variables passed to the simulation
    extra_env = {
        "CLOCK_LEN": str(clk_cycle),                    # Clock period
        "CLOCK_UNIT": clk_cycle_unit,                   # Clock period unit
        "CLOCK_DELAY_INPUT": str(clk_delay_in),         # Input delay
        "CLOCK_DELAY_UNIT_INPUT": clk_delay_unit_in,    # Input delay unit
        "CLOCK_DELAY_OUTPUT": str(clk_delay_out),       # Output delay
        "CLOCK_DELAY_UNIT_OUTPUT": clk_delay_unit_out,  # Output delay unit
        "MODEL_PATH": model_path                        # Path to quantized model
    }
)

## Summary

This notebook demonstrated the complete workflow for deploying a neural network on the OpenEye hardware accelerator:

1. **Training**: Built and trained a CNN on the MNIST dataset
2. **Quantization**: Converted the model to TensorFlow Lite with INT8 quantization
3. **Verification**: Tested the quantized model to ensure accuracy is maintained
4. **Simulation**: Ran a hardware simulation using Cocotb to verify the model works on OpenEye FPGA

The simulation results can be analyzed using the generated waveform files, and the quantized model is ready for deployment on actual OpenEye hardware.